# 🧠 Smart Query Routing Agent System


## Project Objective

The objective of this project is to develop a Single Agent System that intelligently routes user queries to the appropriate tool. The agent performs calculations, keyword extraction, sentiment analysis, web search, real-time API lookups, and LLM reasoning — all with structured JSON output. On top of the individual tools, an **Orchestrator Agent** plans and executes multi-step pipelines across specialist agents, using an LLM-based planner (with a rule-based fallback), a JSON-schema tool-call format, response caching, retry-with-backoff for external calls, and trajectory-level metrics — turning the single agent into a small, observable multi-agent system.

## Problem Statement

In modern AI systems, agents are required to understand user intent and select the appropriate tool to solve a problem. This project demonstrates how an agent can use conditional routing to perform different tasks efficiently, and how that architecture extends into a resilient, observable multi-agent pipeline as capabilities and reliability requirements grow.

## Workflow Diagram

```
                          User Query
                              |
                              ↓
                    Planning Agent (LLM, JSON-schema tool calls)
                     -- falls back to rule-based planner --
                              |
                              ↓
                   Ordered Plan: [{agent, arguments}, ...]
                              |
   -------------------------------------------------------------------------
   |          |          |          |            |          |         |
Calculator  Keyword   Sentiment  Web Search   Exchange   Current    LLM
   Tool     Extractor   Agent    (cached +     Rate       Time     Agent
                                   retried)   (cached +
                                                retried)
   |          |          |          |            |          |         |
   -------------------------------------------------------------------------
                              |
                     Shared context (chains steps)
                              |
                              ↓
                 JSON Output + Trajectory Log (latency, status, agent used)
```

In [20]:
# Import Libraries

import json
import ast
import operator
import os
import re
import time
from datetime import datetime, timezone
from collections import Counter, defaultdict

import requests

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _VADER_AVAILABLE = True
except ImportError:
    _VADER_AVAILABLE = False

In [21]:
# 🛠️ TOOL 1: Calculator
# AST-based evaluator (numbers + basic operators only) instead of eval(), so it
# cannot execute arbitrary code on untrusted input.

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely (numbers and + - * / ** only)."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree.body))
    except Exception:
        return "Error in calculation"

In [22]:
# TOOL 2: Keyword Extractor
# Preserves first-seen order (instead of set(), which does not guarantee order).

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        seen = []
        for w in words:
            lw = w.lower().strip(".,!?")
            if len(lw) > 4 and lw not in seen:
                seen.append(lw)
        return seen[:5]
    except Exception:
        return []

## Tool: Sentiment Analysis

Uses **VADER**, a lexicon/rule-based sentiment model well suited to short, informal text. Falls back to a small built-in lexicon if the package isn't installed, so the agent never crashes for lack of a dependency. Every tool below returns a `status` field (`success` / `fallback` / `error`) — this is what powers the trajectory metrics further down.

In [23]:
# 🛠️ TOOL 3: Sentiment Analysis

_FALLBACK_POSITIVE = {"good", "great", "excellent", "love", "amazing", "happy", "awesome", "fantastic"}
_FALLBACK_NEGATIVE = {"bad", "terrible", "hate", "awful", "sad", "worst", "horrible", "poor"}

if _VADER_AVAILABLE:
    _sentiment_analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment(text: str) -> dict:
    """Return sentiment label + score for a piece of text."""
    try:
        if _VADER_AVAILABLE:
            scores = _sentiment_analyzer.polarity_scores(text)
            compound = scores["compound"]
            if compound >= 0.05:
                label = "positive"
            elif compound <= -0.05:
                label = "negative"
            else:
                label = "neutral"
            return {"label": label, "score": round(compound, 3), "engine": "vader", "status": "success"}
        words = set(w.lower().strip(".,!?") for w in text.split())
        pos = len(words & _FALLBACK_POSITIVE)
        neg = len(words & _FALLBACK_NEGATIVE)
        label = "positive" if pos > neg else "negative" if neg > pos else "neutral"
        return {"label": label, "score": pos - neg, "engine": "fallback_lexicon", "status": "fallback"}
    except Exception as e:
        return {"label": "error", "score": 0, "engine": "none", "status": "error", "error": str(e)}

## Reliability Helpers: Retry-with-Backoff and Caching

Two cross-cutting helpers used by every tool that makes a network call:

- **`with_retries`** — retries a network call up to `max_retries` times with exponential backoff (`base_delay * 2^attempt`) before giving up. This replaces the earlier "single fallback, no retry" behavior and directly implements the retry-loop idea from Q4/Q8.
- **`cache_get` / `cache_set`** — a simple in-memory TTL cache keyed by tool + arguments, so repeated identical web-search or exchange-rate lookups are served instantly instead of re-hitting the network (and, in a paid-API setting, re-incurring cost).

In [24]:
# ---- Retry-with-backoff ----

def with_retries(fn, max_retries: int = 3, base_delay: float = 0.2, retry_on=lambda r: False):
    """Call fn() up to max_retries times with exponential backoff.
    retry_on(result) can flag a 'successful' call (e.g. HTTP 200) that should still be retried
    (e.g. a non-200 status code), not just exceptions."""
    last_exc = None
    for attempt in range(max_retries):
        try:
            result = fn()
            if retry_on(result):
                raise ConnectionError(f"retryable result on attempt {attempt + 1}")
            return result
        except Exception as e:
            last_exc = e
            if attempt < max_retries - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exc


# ---- In-memory TTL cache ----

_CACHE = {}

def cache_get(key, ttl: float):
    entry = _CACHE.get(key)
    if entry and (time.time() - entry["ts"]) < ttl:
        return entry["value"]
    return None

def cache_set(key, value):
    _CACHE[key] = {"value": value, "ts": time.time()}

## Tool: Web Search (cached + retried)

Calls Wikipedia's public search API with retry-with-backoff, and caches results for 5 minutes so a repeated search doesn't hit the network again. If the network call is unavailable (as in this sandboxed environment) or keeps failing after retries, it degrades to a clearly labeled simulated result instead of crashing.

In [25]:
# 🛠️ TOOL 4: Web Search

def web_search(query: str) -> dict:
    """Search the web (Wikipedia) for a query and return the top result summary."""
    cache_key = ("search", query.lower().strip())
    cached = cache_get(cache_key, ttl=300)
    if cached is not None:
        out = dict(cached)
        out["cached"] = True
        return out

    try:
        def _do_request():
            return requests.get(
                "https://en.wikipedia.org/w/api.php",
                params={"action": "query", "list": "search", "srsearch": query,
                        "format": "json", "srlimit": 3},
                timeout=5,
            )
        resp = with_retries(_do_request, max_retries=3, base_delay=0.2,
                             retry_on=lambda r: r.status_code != 200)
        hits = resp.json().get("query", {}).get("search", [])
        results = [{"title": h["title"], "snippet": re.sub("<[^<]+?>", "", h["snippet"])} for h in hits]
        out = {"query": query, "results": results, "source": "wikipedia_live",
               "status": "success", "cached": False}
    except Exception:
        out = {
            "query": query,
            "results": [{"title": "(simulated result)",
                          "snippet": f"Live web search unavailable after retries; placeholder for '{query}'."}],
            "source": "simulated_fallback", "status": "fallback", "cached": False,
        }
    cache_set(cache_key, out)
    return out

## Tool: Real-Time Information Retrieval (cached + retried)

`get_current_time` is genuinely real-time (system clock, no network). `get_exchange_rate` calls a free public API with retry-with-backoff and a 60-second cache, since exchange rates change slowly enough that re-fetching every call would be wasteful.

In [26]:
# 🛠️ TOOL 5: Real-Time Information Retrieval

def get_current_time() -> dict:
    now = datetime.now(timezone.utc)
    return {"utc_time": now.strftime("%Y-%m-%d %H:%M:%S UTC"), "source": "system_clock", "status": "success"}

def get_exchange_rate(base: str = "USD", target: str = "EUR") -> dict:
    """Fetch a live exchange rate from a free public API (no key required)."""
    cache_key = ("fx", base.upper(), target.upper())
    cached = cache_get(cache_key, ttl=60)
    if cached is not None:
        out = dict(cached)
        out["cached"] = True
        return out

    try:
        def _do_request():
            return requests.get(
                "https://api.frankfurter.app/latest",
                params={"from": base.upper(), "to": target.upper()},
                timeout=5,
            )
        resp = with_retries(_do_request, max_retries=3, base_delay=0.2,
                             retry_on=lambda r: r.status_code != 200)
        rate = resp.json()["rates"][target.upper()]
        out = {"base": base.upper(), "target": target.upper(), "rate": rate,
               "source": "frankfurter_live", "status": "success", "cached": False}
    except Exception:
        out = {"base": base.upper(), "target": target.upper(), "rate": None,
               "source": "simulated_fallback", "status": "fallback", "cached": False}
    cache_set(cache_key, out)
    return out

## Tool: LLM Integration

Calls the Anthropic Claude API (with retry-with-backoff) using an `ANTHROPIC_API_KEY` environment variable — never hard-code API keys in a notebook. If no key is configured, or the request keeps failing after retries, it falls back to a simple simulated response so the notebook still runs end-to-end without a paid API key. This same function is reused by the **planning agent** below.

In [27]:
# 🛠️ TOOL 6: LLM Integration (Anthropic Claude)

def call_llm(prompt: str) -> dict:
    """Call an LLM for open-ended reasoning; falls back to a simulated response if no API key/network."""
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        try:
            def _do_request():
                return requests.post(
                    "https://api.anthropic.com/v1/messages",
                    headers={"x-api-key": api_key, "anthropic-version": "2023-06-01",
                             "content-type": "application/json"},
                    json={"model": "claude-sonnet-4-6", "max_tokens": 300,
                          "messages": [{"role": "user", "content": prompt}]},
                    timeout=15,
                )
            resp = with_retries(_do_request, max_retries=3, base_delay=0.3,
                                 retry_on=lambda r: r.status_code != 200)
            content = resp.json()["content"]
            text = "".join(b.get("text", "") for b in content if b.get("type") == "text")
            return {"answer": text, "source": "claude_live", "status": "success"}
        except Exception:
            pass  # fall through to simulated response below

    sentences = re.split(r'(?<=[.!?]) +', prompt.strip())
    simulated = sentences[0] if sentences else prompt
    return {"answer": f"[Simulated LLM response — no ANTHROPIC_API_KEY configured] {simulated}",
            "source": "simulated_fallback", "status": "fallback"}

## JSON-Schema Tool Calls + LLM Planner (replaces the "and"/"then" splitter)

Instead of splitting a query on connector words, each tool is described as a **JSON schema** (name, description, parameters) — the same style used by real tool-calling LLM APIs. `llm_plan` asks the LLM to return *only* a JSON array of `{"agent": ..., "arguments": {...}}` objects that conform to these schemas — the LLM itself decides which agents to call and with what arguments. If no API key is configured (so `call_llm` returns a simulated, non-JSON answer) or the response can't be parsed as a valid plan, `plan_pipeline` transparently falls back to `rule_based_plan`, which produces the same structured `{agent, arguments}` format using keyword rules — so the rest of the system doesn't care which planner produced the plan.

In [28]:
# ---- Tool schemas (JSON-schema style, like a real tool-calling API) ----

TOOL_SCHEMAS = [
    {"name": "calculator", "description": "Evaluate a math expression.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                     "required": ["expression"]}},
    {"name": "keywords", "description": "Extract top keywords from a block of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
    {"name": "sentiment", "description": "Analyze the sentiment of a piece of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
    {"name": "web_search", "description": "Search the web for information about a topic.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "exchange_rate", "description": "Get a live currency exchange rate.",
     "parameters": {"type": "object",
                     "properties": {"base": {"type": "string"}, "target": {"type": "string"}},
                     "required": ["base", "target"]}},
    {"name": "current_time", "description": "Get the current UTC time.",
     "parameters": {"type": "object", "properties": {}}},
    {"name": "llm", "description": ("Use an LLM for reasoning, summarization, or explanation. "
                                       "Use the literal string '{previous_result}' in the prompt "
                                       "to refer to the previous step's output."),
     "parameters": {"type": "object", "properties": {"prompt": {"type": "string"}}, "required": ["prompt"]}},
    {"name": "general", "description": "Fallback for anything that doesn't match another tool.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
]

def _extract_json_array(text: str):
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON array found in LLM response")
    return json.loads(match.group(0))

def llm_plan(query: str):
    """Ask the LLM to plan the tool-call pipeline itself. Returns None if unavailable/unparseable."""
    planning_prompt = (
        "You are a planning agent. Given the tool schemas below and a user query, output ONLY a JSON "
        "array of tool calls needed to answer it, each of the form "
        '{"agent": "<tool name>", "arguments": {...}}. Use multiple entries if the query needs '
        "multiple steps. Output nothing except the JSON array.\n\n"
        f"Tool schemas: {json.dumps(TOOL_SCHEMAS)}\n\nQuery: {query}"
    )
    llm_result = call_llm(planning_prompt)
    if llm_result["source"] != "claude_live":
        return None  # no real LLM available -> let the caller fall back to rule_based_plan
    try:
        return _extract_json_array(llm_result["answer"])
    except Exception:
        return None

def rule_based_plan(query: str):
    """Deterministic fallback planner: same {agent, arguments} schema, built from keyword rules."""
    parts = re.split(r"\s+and then\s+|\s+then\s+|\s+and\s+", query, flags=re.IGNORECASE)
    steps = [p.strip() for p in parts if p.strip()]
    plan = []
    for step in steps:
        q = step.lower()
        if "calculate" in q:
            plan.append({"agent": "calculator", "arguments": {"expression": q.replace("calculate", "").strip()}})
        elif "keywords" in q:
            text = step.lower().replace("extract keywords from", "").strip()
            plan.append({"agent": "keywords", "arguments": {"text": text}})
        elif "sentiment" in q:
            text = re.sub(r"what is the sentiment of|analyze sentiment|sentiment of", "", q).strip() or step
            plan.append({"agent": "sentiment", "arguments": {"text": text}})
        elif "search" in q:
            text = q.replace("search for", "").replace("search", "").strip()
            plan.append({"agent": "web_search", "arguments": {"query": text}})
        elif "exchange rate" in q:
            plan.append({"agent": "exchange_rate", "arguments": {"base": "USD", "target": "EUR"}})
        elif "time" in q:
            plan.append({"agent": "current_time", "arguments": {}})
        elif any(w in q for w in ["explain", "summarize", "why"]):
            plan.append({"agent": "llm", "arguments": {"prompt": "{previous_result}" if plan else step}})
        else:
            plan.append({"agent": "general", "arguments": {"text": step}})
    return plan

def plan_pipeline(query: str):
    """Try the LLM planner first; fall back to the rule-based planner. Returns (plan, plan_source)."""
    plan = llm_plan(query)
    if plan:
        return plan, "llm_planner"
    return rule_based_plan(query), "rule_based_planner"

## Orchestrator: Executes the Plan, Chains Context, Logs Trajectory Metrics

Each tool is wrapped in a small executor function that takes structured `arguments` (from the plan) plus a shared `context` dict, so a later step (e.g. `llm`) can reference an earlier step's output (e.g. `web_search`) via `{previous_result}`. Every step is timed and logged to `TRAJECTORY_LOG` with its agent name, latency, and status — directly implementing the trajectory-evaluation and cost/completion-rate metrics from Q9/Q10.

In [29]:
# ---- Executors: adapt each tool to the (arguments, context) -> result interface ----

def _exec_web_search(args, ctx):
    result = web_search(args.get("query", ""))
    summary = " ".join(r["snippet"] for r in result["results"]) or args.get("query", "")
    ctx["last_text"] = summary
    return result

def _exec_llm(args, ctx):
    prompt = args.get("prompt") or ""
    if "{previous_result}" in prompt or not prompt:
        prev = ctx.get("last_text", "")
        prompt = prompt.replace("{previous_result}", prev) if prompt else f"Summarize this: {prev}"
    result = call_llm(prompt)
    ctx["last_text"] = result["answer"]
    return result

def _exec_sentiment(args, ctx):
    text = args.get("text") or ctx.get("last_text", "")
    return analyze_sentiment(text)

AGENT_EXECUTORS = {
    "calculator": lambda args, ctx: {"result": calculator(args.get("expression", "")), "status": "success"},
    "keywords": lambda args, ctx: {"result": extract_keywords(args.get("text", "")), "status": "success"},
    "sentiment": _exec_sentiment,
    "web_search": _exec_web_search,
    "exchange_rate": lambda args, ctx: get_exchange_rate(args.get("base", "USD"), args.get("target", "EUR")),
    "current_time": lambda args, ctx: get_current_time(),
    "llm": _exec_llm,
    "general": lambda args, ctx: {"result": f"You asked: {args.get('text', '')}", "status": "success"},
}

In [30]:
# ---- Trajectory metrics (Q9/Q10): which agents were used, latency, completion rate ----

TRAJECTORY_LOG = []

def log_step(run_id, agent, latency_ms, status):
    TRAJECTORY_LOG.append({
        "run_id": run_id, "agent": agent, "latency_ms": latency_ms, "status": status,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    })

def get_trajectory_metrics() -> dict:
    if not TRAJECTORY_LOG:
        return {"total_steps": 0}
    total = len(TRAJECTORY_LOG)
    completed = sum(1 for r in TRAJECTORY_LOG if r["status"] in ("success", "fallback"))
    by_agent = defaultdict(list)
    for r in TRAJECTORY_LOG:
        by_agent[r["agent"]].append(r["latency_ms"])
    avg_latency = {a: round(sum(v) / len(v), 2) for a, v in by_agent.items()}
    usage_counts = dict(Counter(r["agent"] for r in TRAJECTORY_LOG))
    live_calls = sum(1 for r in TRAJECTORY_LOG
                      if r["status"] == "success" and r["agent"] in ("web_search", "exchange_rate", "llm"))
    return {
        "total_steps": total,
        "task_completion_rate": round(completed / total, 3),   # Q10: completion rate
        "agent_usage_counts": usage_counts,
        "avg_latency_ms_by_agent": avg_latency,
        "live_external_calls": live_calls,                     # Q10: rough cost proxy
    }

In [31]:
# ---- Orchestrator: plan -> execute -> log ----

class OrchestratorAgent:
    """
    1. Query Analysis + Planning — plan_pipeline() asks the LLM planner (JSON-schema tool calls) or,
       if that's unavailable, the rule-based planner, to produce an ordered [{agent, arguments}, ...].
    2. Conditional Routing — each planned step is dispatched to its executor via AGENT_EXECUTORS.
    3. Shared Context — a dict passed between steps so later agents can use earlier results.
    4. Trajectory Logging — every step's latency and status is recorded for later analysis.
    """

    def __init__(self):
        self.run_counter = 0

    def run(self, query: str) -> str:
        self.run_counter += 1
        run_id = self.run_counter
        plan, plan_source = plan_pipeline(query)
        context = {}
        trace = []
        for step in plan:
            agent_name = step.get("agent", "general")
            arguments = step.get("arguments", {}) or {}
            executor = AGENT_EXECUTORS.get(agent_name, AGENT_EXECUTORS["general"])
            start = time.perf_counter()
            try:
                output = executor(arguments, context)
                status = output.get("status", "success")
            except Exception as e:
                output = {"result": None, "error": str(e)}
                status = "error"
            latency_ms = round((time.perf_counter() - start) * 1000, 2)
            log_step(run_id, agent_name, latency_ms, status)
            trace.append({"agent": agent_name, "arguments": arguments, "output": output,
                          "latency_ms": latency_ms, "status": status})
        response = {
            "type": "multi_agent" if len(trace) > 1 else (trace[0]["agent"] if trace else "none"),
            "plan_source": plan_source,
            "steps_executed": len(trace),
            "trace": trace,
        }
        return json.dumps(response)

orchestrator = OrchestratorAgent()

## Error Handling

Every tool wraps its logic in try/except. Network-dependent tools (web search, exchange rate, LLM) now retry up to 3 times with exponential backoff before degrading to a clearly labeled `simulated_fallback` result, rather than failing on the first network hiccup (Q4/Q8). The orchestrator itself also catches any executor exception per-step and logs it as `status: "error"` without stopping the rest of the run.

## Testing the Orchestrator (single-step queries)

Since no `ANTHROPIC_API_KEY` is configured in this notebook by default, `plan_pipeline` will fall back to `rule_based_planner` — the printed `plan_source` field shows this explicitly.

In [32]:
single_step_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 10 / 0",
    "What is the sentiment of I absolutely love this new phone",
    "Search for the Eiffel Tower",
    "What time is it",
]

for q in single_step_queries:
    print("Query :", q)
    print("Response :", orchestrator.run(q))
    print("-" * 50)

Query : Calculate 20 + 5
Response : {"type": "calculator", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "calculator", "arguments": {"expression": "20 + 5"}, "output": {"result": "25", "status": "success"}, "latency_ms": 0.03, "status": "success"}]}
--------------------------------------------------
Query : Extract keywords from Artificial Intelligence is transforming industries
Response : {"type": "keywords", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "keywords", "arguments": {"text": "artificial intelligence is transforming industries"}, "output": {"result": ["artificial", "intelligence", "transforming", "industries"], "status": "success"}, "latency_ms": 0.01, "status": "success"}]}
--------------------------------------------------
Query : What is machine learning?
Response : {"type": "general", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "general", "arguments": {"text": "What is mac

## Self-Contained Version of the Single-Step Test Cell

The cell below duplicates every tool, the planner, and the orchestrator inline, so it can be copied out and run in a brand-new notebook on its own — without needing any of the earlier setup cells in this file to be executed first.

In [37]:
import json
import ast
import operator
import os
import re
import time
from datetime import datetime, timezone
from collections import Counter, defaultdict

import requests

try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _VADER_AVAILABLE = True
except ImportError:
    _VADER_AVAILABLE = False

# ---- TOOL 1: Calculator ----
_ALLOWED_OPERATORS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPERATORS:
        return _ALLOWED_OPERATORS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Unsupported expression")

def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree.body))
    except Exception:
        return "Error in calculation"

# ---- TOOL 2: Keyword Extractor ----
def extract_keywords(text: str) -> list:
    try:
        words = text.split()
        seen = []
        for w in words:
            lw = w.lower().strip(".,!?")
            if len(lw) > 4 and lw not in seen:
                seen.append(lw)
        return seen[:5]
    except Exception:
        return []

# ---- TOOL 3: Sentiment ----
_FALLBACK_POSITIVE = {"good", "great", "excellent", "love", "amazing", "happy", "awesome", "fantastic"}
_FALLBACK_NEGATIVE = {"bad", "terrible", "hate", "awful", "sad", "worst", "horrible", "poor"}

if _VADER_AVAILABLE:
    _sentiment_analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment(text: str) -> dict:
    try:
        if _VADER_AVAILABLE:
            scores = _sentiment_analyzer.polarity_scores(text)
            compound = scores["compound"]
            if compound >= 0.05:
                label = "positive"
            elif compound <= -0.05:
                label = "negative"
            else:
                label = "neutral"
            return {"label": label, "score": round(compound, 3), "engine": "vader", "status": "success"}
        words = set(w.lower().strip(".,!?") for w in text.split())
        pos = len(words & _FALLBACK_POSITIVE)
        neg = len(words & _FALLBACK_NEGATIVE)
        label = "positive" if pos > neg else "negative" if neg > pos else "neutral"
        return {"label": label, "score": pos - neg, "engine": "fallback_lexicon", "status": "fallback"}
    except Exception as e:
        return {"label": "error", "score": 0, "engine": "none", "status": "error", "error": str(e)}

# ---- Retry-with-backoff + TTL cache ----
def with_retries(fn, max_retries: int = 3, base_delay: float = 0.2, retry_on=lambda r: False):
    last_exc = None
    for attempt in range(max_retries):
        try:
            result = fn()
            if retry_on(result):
                raise ConnectionError(f"retryable result on attempt {attempt + 1}")
            return result
        except Exception as e:
            last_exc = e
            if attempt < max_retries - 1:
                time.sleep(base_delay * (2 ** attempt))
    raise last_exc

_CACHE = {}

def cache_get(key, ttl: float):
    entry = _CACHE.get(key)
    if entry and (time.time() - entry["ts"]) < ttl:
        return entry["value"]
    return None

def cache_set(key, value):
    _CACHE[key] = {"value": value, "ts": time.time()}

# ---- TOOL 4: Web Search ----
def web_search(query: str) -> dict:
    cache_key = ("search", query.lower().strip())
    cached = cache_get(cache_key, ttl=300)
    if cached is not None:
        out = dict(cached)
        out["cached"] = True
        return out
    try:
        def _do_request():
            resp = requests.get(
                "https://en.wikipedia.org/w/api.php",
                params={"action": "query", "list": "search", "srsearch": query,
                        "format": "json", "srlimit": 3},
                timeout=5,
            )
            print(f"Web search API response status code: {resp.status_code}") # Added print statement
            return resp
        resp = with_retries(_do_request, max_retries=3, base_delay=0.2,
                             retry_on=lambda r: r.status_code != 200)
        hits = resp.json().get("query", {}).get("search", [])
        results = [{"title": h["title"], "snippet": re.sub("<[^<]+?>", "", h["snippet"])} for h in hits]
        out = {"query": query, "results": results, "source": "wikipedia_live",
               "status": "success", "cached": False}
    except Exception:
        out = {
            "query": query,
            "results": [{"title": "(simulated result)",
                         "snippet": f"Live web search unavailable after retries; placeholder for '{query}'."}],
            "source": "simulated_fallback", "status": "fallback", "cached": False,
        }
    cache_set(cache_key, out)
    return out

# ---- TOOL 5: Real-Time Information ----
def get_current_time() -> dict:
    now = datetime.now(timezone.utc)
    return {"utc_time": now.strftime("%Y-%m-%d %H:%M:%S UTC"), "source": "system_clock", "status": "success"}

def get_exchange_rate(base: str = "USD", target: str = "EUR") -> dict:
    cache_key = ("fx", base.upper(), target.upper())
    cached = cache_get(cache_key, ttl=60)
    if cached is not None:
        out = dict(cached)
        out["cached"] = True
        return out
    try:
        def _do_request():
            return requests.get(
                "https://api.frankfurter.app/latest",
                params={"from": base.upper(), "to": target.upper()},
                timeout=5,
            )
        resp = with_retries(_do_request, max_retries=3, base_delay=0.2,
                             retry_on=lambda r: r.status_code != 200)
        rate = resp.json()["rates"][target.upper()]
        out = {"base": base.upper(), "target": target.upper(), "rate": rate,
               "source": "frankfurter_live", "status": "success", "cached": False}
    except Exception:
        out = {"base": base.upper(), "target": target.upper(), "rate": None,
               "source": "simulated_fallback", "status": "fallback", "cached": False}
    cache_set(cache_key, out)
    return out

# ---- TOOL 6: LLM Integration ----
def call_llm(prompt: str) -> dict:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if api_key:
        try:
            def _do_request():
                return requests.post(
                    "https://api.anthropic.com/v1/messages",
                    headers={"x-api-key": api_key, "anthropic-version": "2023-06-01",
                             "content-type": "application/json"},
                    json={"model": "claude-sonnet-4-6", "max_tokens": 300,
                          "messages": [{"role": "user", "content": prompt}]},
                    timeout=15,
                )
            resp = with_retries(_do_request, max_retries=3, base_delay=0.3,
                                 retry_on=lambda r: r.status_code != 200)
            content = resp.json()["content"]
            text = "".join(b.get("text", "") for b in content if b.get("type") == "text")
            return {"answer": text, "source": "claude_live", "status": "success"}
        except Exception:
            pass
    sentences = re.split(r'(?<=[.!?]) +', prompt.strip())
    simulated = sentences[0] if sentences else prompt
    return {"answer": f"[Simulated LLM response \u2014 no ANTHROPIC_API_KEY configured] {simulated}",
            "source": "simulated_fallback", "status": "fallback"}

# ---- JSON-schema tool definitions + planner ----
TOOL_SCHEMAS = [
    {"name": "calculator", "description": "Evaluate a math expression.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}},
                    "required": ["expression"]}},
    {"name": "keywords", "description": "Extract top keywords from a block of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
    {"name": "sentiment", "description": "Analyze the sentiment of a piece of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
    {"name": "web_search", "description": "Search the web for information about a topic.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "exchange_rate", "description": "Get a live currency exchange rate.",
     "parameters": {"type": "object",
                    "properties": {"base": {"type": "string"}, "target": {"type": "string"}},
                    "required": ["base", "target"]}},
    {"name": "current_time", "description": "Get the current UTC time.",
     "parameters": {"type": "object", "properties": {}}},
    {"name": "llm", "description": ("Use an LLM for reasoning, summarization, or explanation. "
                                    "Use the literal string '{previous_result}' in the prompt "
                                    "to refer to the previous step's output."),
     "parameters": {"type": "object", "properties": {"prompt": {"type": "string"}}, "required": ["prompt"]}},
    {"name": "general", "description": "Fallback for anything that doesn't match another tool.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
]

def _extract_json_array(text: str):
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON array found in LLM response")
    return json.loads(match.group(0))

def llm_plan(query: str):
    planning_prompt = (
        "You are a planning agent. Given the tool schemas below and a user query, output ONLY a JSON "
        "array of tool calls needed to answer it, each of the form "
        '{"agent": "<tool name>", "arguments": {...}}. Use multiple entries if the query needs '
        "multiple steps. Output nothing except the JSON array.\n\n"
        f"Tool schemas: {json.dumps(TOOL_SCHEMAS)}\n\nQuery: {query}"
    )
    llm_result = call_llm(planning_prompt)
    if llm_result["source"] != "claude_live":
        return None
    try:
        return _extract_json_array(llm_result["answer"])
    except Exception:
        return None

def rule_based_plan(query: str):
    parts = re.split(r"\s+and then\s+|\s+then\s+|\s+and\s+", query, flags=re.IGNORECASE)
    steps = [p.strip() for p in parts if p.strip()]
    plan = []
    for step in steps:
        q = step.lower()
        if "calculate" in q:
            plan.append({"agent": "calculator", "arguments": {"expression": q.replace("calculate", "").strip()}})
        elif "keywords" in q:
            text = step.lower().replace("extract keywords from", "").strip()
            plan.append({"agent": "keywords", "arguments": {"text": text}})
        elif "sentiment" in q:
            text = re.sub(r"what is the sentiment of|analyze sentiment|sentiment of", "", q).strip() or step
            plan.append({"agent": "sentiment", "arguments": {"text": text}})
        elif "search" in q:
            text = q.replace("search for", "").replace("search", "").strip()
            plan.append({"agent": "web_search", "arguments": {"query": text}})
        elif "exchange rate" in q:
            plan.append({"agent": "exchange_rate", "arguments": {"base": "USD", "target": "EUR"}})
        elif "time" in q:
            plan.append({"agent": "current_time", "arguments": {}})
        elif any(w in q for w in ["explain", "summarize", "why"]):
            plan.append({"agent": "llm", "arguments": {"prompt": "{previous_result}" if plan else step}})
        else:
            plan.append({"agent": "general", "arguments": {"text": step}})
    return plan

def plan_pipeline(query: str):
    plan = llm_plan(query)
    if plan:
        return plan, "llm_planner"
    return rule_based_plan(query), "rule_based_planner"

# ---- Executors ----
def _exec_web_search(args, ctx):
    result = web_search(args.get("query", ""))
    summary = " ".join(r["snippet"] for r in result["results"]) or args.get("query", "")
    ctx["last_text"] = summary
    return result

def _exec_llm(args, ctx):
    prompt = args.get("prompt") or ""
    if "{previous_result}" in prompt or not prompt:
        prev = ctx.get("last_text", "")
        prompt = prompt.replace("{previous_result}", prev) if prompt else f"Summarize this: {prev}"
    result = call_llm(prompt)
    ctx["last_text"] = result["answer"]
    return result

def _exec_sentiment(args, ctx):
    text = args.get("text") or ctx.get("last_text", "")
    return analyze_sentiment(text)

AGENT_EXECUTORS = {
    "calculator": lambda args, ctx: {"result": calculator(args.get("expression", "")), "status": "success"},
    "keywords": lambda args, ctx: {"result": extract_keywords(args.get("text", "")), "status": "success"},
    "sentiment": _exec_sentiment,
    "web_search": _exec_web_search,
    "exchange_rate": lambda args, ctx: get_exchange_rate(args.get("base", "USD"), args.get("target", "EUR")),
    "current_time": lambda args, ctx: get_current_time(),
    "llm": _exec_llm,
    "general": lambda args, ctx: {"result": f"You asked: {args.get('text', '')}", "status": "success"},
}

# ---- Trajectory metrics ----
TRAJECTORY_LOG = []

def log_step(run_id, agent, latency_ms, status):
    TRAJECTORY_LOG.append({
        "run_id": run_id, "agent": agent, "latency_ms": latency_ms, "status": status,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    })

def get_trajectory_metrics() -> dict:
    if not TRAJECTORY_LOG:
        return {"total_steps": 0}
    total = len(TRAJECTORY_LOG)
    completed = sum(1 for r in TRAJECTORY_LOG if r["status"] in ("success", "fallback"))
    by_agent = defaultdict(list)
    for r in TRAJECTORY_LOG:
        by_agent[r["agent"]].append(r["latency_ms"])
    avg_latency = {a: round(sum(v) / len(v), 2) for a, v in by_agent.items()}
    usage_counts = dict(Counter(r["agent"] for r in TRAJECTORY_LOG))
    live_calls = sum(1 for r in TRAJECTORY_LOG
                      if r["status"] == "success" and r["agent"] in ("web_search", "exchange_rate", "llm"))
    return {
        "total_steps": total,
        "task_completion_rate": round(completed / total, 3),
        "agent_usage_counts": usage_counts,
        "avg_latency_ms_by_agent": avg_latency,
        "live_external_calls": live_calls,
    }

# ---- Orchestrator ----
class OrchestratorAgent:
    def __init__(self):
        self.run_counter = 0

    def run(self, query: str) -> str:
        self.run_counter += 1
        run_id = self.run_counter
        plan, plan_source = plan_pipeline(query)
        context = {}
        trace = []
        for step in plan:
            agent_name = step.get("agent", "general")
            arguments = step.get("arguments", {}) or {}
            executor = AGENT_EXECUTORS.get(agent_name, AGENT_EXECUTORS["general"])
            start = time.perf_counter()
            try:
                output = executor(arguments, context)
                status = output.get("status", "success")
            except Exception as e:
                output = {"result": None, "error": str(e)}
                status = "error"
            latency_ms = round((time.perf_counter() - start) * 1000, 2)
            log_step(run_id, agent_name, latency_ms, status)
            trace.append({"agent": agent_name, "arguments": arguments, "output": output,
                          "latency_ms": latency_ms, "status": status})
        response = {
            "type": "multi_agent" if len(trace) > 1 else (trace[0]["agent"] if trace else "none"),
            "plan_source": plan_source,
            "steps_executed": len(trace),
            "trace": trace,
        }
        return json.dumps(response)

orchestrator = OrchestratorAgent()

# ---- Test cases ----
single_step_queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 10 / 0",
    "What is the sentiment of I absolutely love this new phone",
    "Search for the Eiffel Tower",
    "What time is it",
]

for q in single_step_queries:
    print("Query :", q)
    print("Response :", orchestrator.run(q))
    print("-" * 50)


Query : Calculate 20 + 5
Response : {"type": "calculator", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "calculator", "arguments": {"expression": "20 + 5"}, "output": {"result": "25", "status": "success"}, "latency_ms": 0.04, "status": "success"}]}
--------------------------------------------------
Query : Extract keywords from Artificial Intelligence is transforming industries
Response : {"type": "keywords", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "keywords", "arguments": {"text": "artificial intelligence is transforming industries"}, "output": {"result": ["artificial", "intelligence", "transforming", "industries"], "status": "success"}, "latency_ms": 0.01, "status": "success"}]}
--------------------------------------------------
Query : What is machine learning?
Response : {"type": "general", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "general", "arguments": {"text": "What is mac

In [34]:
try:
    response = requests.get("https://www.google.com", timeout=5)
    if response.status_code == 200:
        print("Network connectivity is working. You should be able to perform live web searches.")
    else:
        print(f"Failed to reach Google.com with status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Network connectivity error: {e}\n\nIt seems there might be an issue with network access from this environment, or a temporary problem. Live web search won't work until this is resolved.")

Network connectivity is working. You should be able to perform live web searches.


If the network test passes, and `web_search` still returns simulated results, it might be a temporary issue with the Wikipedia API itself, or a specific block. However, usually, a working network connection is all that's needed for the Wikipedia search to function live.

## Testing Multi-Step Pipelines and the Search Cache

The second query needs Web Search chained into the LLM Agent. The last two queries are identical, so the second one should show `"cached": true` and a much lower `latency_ms` for the `web_search` step, proving the cache is working.

In [35]:
complex_queries = [
    "Calculate 15 * 3",
    "Search for the Great Barrier Reef and summarize it",
    "What is the sentiment of this service is absolutely terrible and then explain why",
    "Search for the Eiffel Tower",
    "Search for the Eiffel Tower",
]

for q in complex_queries:
    print("Query :", q)
    print("Response :", orchestrator.run(q))
    print("=" * 60)

Query : Calculate 15 * 3
Response : {"type": "calculator", "plan_source": "rule_based_planner", "steps_executed": 1, "trace": [{"agent": "calculator", "arguments": {"expression": "15 * 3"}, "output": {"result": "45", "status": "success"}, "latency_ms": 0.03, "status": "success"}]}
Query : Search for the Great Barrier Reef and summarize it
Web search API response status code: 403
Web search API response status code: 403
Web search API response status code: 403
Response : {"type": "multi_agent", "plan_source": "rule_based_planner", "steps_executed": 2, "trace": [{"agent": "web_search", "arguments": {"query": "the great barrier reef"}, "output": {"query": "the great barrier reef", "results": [{"title": "(simulated result)", "snippet": "Live web search unavailable after retries; placeholder for 'the great barrier reef'."}], "source": "simulated_fallback", "status": "fallback", "cached": false}, "latency_ms": 674.47, "status": "fallback"}, {"agent": "llm", "arguments": {"prompt": "{previous

## Trajectory Metrics (Q9/Q10)

After running the queries above, `get_trajectory_metrics()` reports task completion rate, how often each agent was used, average latency per agent, and a rough "cost" proxy (count of live external calls) — exactly the metrics described in Q9 (trajectory evaluation) and Q10 (completion rate & cost).

In [36]:
print(json.dumps(get_trajectory_metrics(), indent=2))

{
  "total_steps": 14,
  "task_completion_rate": 1.0,
  "agent_usage_counts": {
    "calculator": 3,
    "keywords": 1,
    "general": 1,
    "sentiment": 2,
    "web_search": 4,
    "current_time": 1,
    "llm": 2
  },
  "avg_latency_ms_by_agent": {
    "calculator": 0.03,
    "keywords": 0.01,
    "general": 0.0,
    "sentiment": 0.01,
    "web_search": 340.38,
    "current_time": 0.02,
    "llm": 0.04
  },
  "live_external_calls": 0
}


## Conclusion

This project evolved from a two-tool keyword router into a small, observable multi-agent system: six specialist tools (calculator, keywords, sentiment, web search, real-time API, LLM), each describable as a JSON schema; an LLM-based planning agent (with a deterministic rule-based fallback) that decides the tool-call pipeline itself instead of splitting on "and"/"then"; an in-memory TTL cache for repeated search and exchange-rate lookups; retry-with-backoff around every external call; and a trajectory log that reports task completion rate, per-agent latency, and external-call volume. Together these demonstrate conditional routing, cycles/retries, shared state between nodes, JSON-schema tool structuring, and full trajectory-level observability — the core concepts covered in the quiz.